In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"
import keras
import torch

In [2]:
import pickle
import tarfile
import datetime
import numpy as np
import pandas as pd
import urllib.request
import sklearn.metrics
import matplotlib.pyplot as plt
import zipfile

from pathlib import Path
import cv2
from typing import Literal

In [3]:
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.is_available())

CUDA: 12.8
GPU: True


In [4]:
%load_ext tensorboard

In [5]:
BATCH_SIZE = 256
LOGS_DIR = '../logs'
DATA_DIR = '../data'

os.makedirs(LOGS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

In [6]:
def download_data():
    filepath = os.path.join(DATA_DIR, 'tiny-imagenet-200.zip')
    if not os.path.exists('tiny-imagenet-200'):
        print("Donwloading...")
        urllib.request.urlretrieve('http://cs231n.stanford.edu/tiny-imagenet-200.zip', filepath)

        print("Extracting...")
        file = zipfile.ZipFile(filepath, 'r')
        file.extractall(DATA_DIR)

In [7]:
# download_data()

In [8]:
from joblib import Parallel, delayed

In [9]:
class TinyImageNetDataset(keras.utils.PyDataset):

    def __init__(
        self,
        path: str,
        mode: Literal["train", "val"],
        *,
        batch_size=32,
        shuffle=True,
        seed=None,
        prefetch=True,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.batch_size = batch_size
        self.rng = np.random.default_rng(seed)
        self.shuffle = shuffle
        self.prefetch = prefetch

        base_path = Path(path)

        if mode == "train":
            self.read_train(base_path)
        else:
            self.read_val(base_path)

        self.on_epoch_end()

    def read_train(self, base_path: Path):
        train_path = base_path / "train"

        classes, class_mapping = self.get_class_mapping(train_path)

        self.image_paths: list[str] = []
        labels: list[int] = []

        for img_class in classes:
            img_dir = train_path / img_class / "images"

            for img_file in img_dir.glob("*.JPEG"):
                self.image_paths.append(str(img_file))
                labels.append(class_mapping[img_class])

        self.indices = np.arange(len(self.image_paths))
        self.labels = np.array(labels)

        if self.prefetch:
            self.images_data = np.array(
                Parallel(n_jobs=-1)(
                    delayed(cv2.imread)(path) for path in self.image_paths
                )
            )

    def read_val(self, base_path: Path):
        _, mapping = self.get_class_mapping(base_path / "train")
        val_path = base_path / "val"

        df = pd.read_csv(val_path / "val_annotations.txt", delimiter="\t", header=None)
        image_dir = val_path / "images"

        self.image_paths = []
        labels = []

        for index, row in df.iterrows():
            self.image_paths.append(str(image_dir / row[0]))
            labels.append(mapping[row[1]])

        self.labels = np.array(labels)
        self.indices = np.arange(len(self.image_paths))
        if self.prefetch:
            self.images_data = np.array(
                Parallel(n_jobs=-1)(
                    delayed(cv2.imread)(path) for path in self.image_paths
                )
            )

    def get_class_mapping(self, train_path: Path) -> tuple[list[str], dict[str, int]]:
        classes: list[str] = []
        for item in train_path.iterdir():
            if item.is_dir():
                classes.append(item.name)

        return classes, {
            class_name: idx for idx, class_name in enumerate(sorted(classes))
        }

    def change_batch_size(self, new_batch_size):
        self.batch_size = new_batch_size

    def __len__(self):
        return int(np.ceil(len(self.image_paths) / self.batch_size))

    def __getitem__(self, idx):
        start = idx * self.batch_size
        end = min(start + self.batch_size, len(self.image_paths))
        batch_indices = self.indices[start:end]

        if self.prefetch:
            images = self.images_data[batch_indices]
        else:
            images = np.array([cv2.imread(self.image_paths[i]) for i in batch_indices])

        return (
            images,
            self.labels[batch_indices],
        )

    def on_epoch_end(self):
        if self.shuffle:
            self.rng.shuffle(self.indices)

In [10]:
imagenet_path = "../data/tiny-imagenet-200"

In [11]:
train_dataset = TinyImageNetDataset(
    imagenet_path,
    "train",
    batch_size=BATCH_SIZE
)

In [46]:
val_dataset = TinyImageNetDataset(
    imagenet_path,
    "val",
    shuffle=False,
    batch_size=BATCH_SIZE
)

In [13]:
input_shape = (64, 64, 3)
num_classes = 200

In [14]:
x = inputs = keras.Input(shape=input_shape)

x = keras.layers.Rescaling(1./255)(x)

x = keras.layers.Conv2D(32, 3, padding="same", kernel_initializer="he_normal")(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.ReLU()(x)
x = keras.layers.MaxPooling2D(2)(x)

x = keras.layers.Conv2D(64, 3, padding="same", kernel_initializer="he_normal")(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.ReLU()(x)
x = keras.layers.Conv2D(64, 3, padding="same", kernel_initializer="he_normal")(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.ReLU()(x)
x = keras.layers.MaxPooling2D(2)(x)

x = keras.layers.Conv2D(128, 3, padding="same", kernel_initializer="he_normal")(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.ReLU()(x)
x = keras.layers.Conv2D(128, 3, padding="same", kernel_initializer="he_normal")(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.ReLU()(x)
x = keras.layers.MaxPooling2D(2)(x)

x = keras.layers.Conv2D(256, 3, padding="same", kernel_initializer="he_normal")(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.ReLU()(x)
x = keras.layers.Conv2D(256, 3, padding="same", kernel_initializer="he_normal")(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.ReLU()(x)
x = keras.layers.GlobalAveragePooling2D()(x)

x = keras.layers.Dense(256, kernel_initializer="he_normal")(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.ReLU()(x)
x = keras.layers.Dropout(0.3)(x)
x = keras.layers.Dense(num_classes, activation="softmax", dtype="float32")(x)

c:\Users\Yanovich\Documents\Projects\Cats\neural\.venv\Lib\site-packages\keras\src\backend\torch\core.py:243: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:39.)
  return torch.as_tensor(x, dtype=dtype, device=get_device())


In [15]:
model = keras.models.Model(inputs=inputs, outputs=x)

In [16]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 64, 64, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_3 (ReLU)                  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 16, 16, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 16, 16, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_4 (ReLU)                  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 8, 8, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 8, 8, 256)      │         1,024 │
│ (BatchNormalization)            │                        │             

 Total params: 1,284,936 (4.90 MB)

 Trainable params: 1,282,568 (4.89 MB)

 Non-trainable params: 2,368 (9.25 KB)

In [17]:
model.compile(
    optimizer="adam",
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [18]:
logdir = os.path.join(LOGS_DIR, "tiny_imagenet", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))

In [19]:
# model_checkpoint_callback = keras.callbacks.ModelCheckpoint(
#     os.path.join(logdir, 'model.keras'),
#     save_best_only=True
# )

In [20]:
# tensorboard_callback = keras.callbacks.TensorBoard(
#     os.path.join(logdir, 'logs'),    
# )

In [21]:
lr_scheduler = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5, verbose=1
)

In [22]:
# %tensorboard --logdir $logdir

In [23]:
model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    callbacks=[lr_scheduler],
)

Epoch 1/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 49s 123ms/step - accuracy: 0.0868 - loss: 4.4251 - val_accuracy: 0.0755 - val_loss: 4.5908 - learning_rate: 0.0010
Epoch 2/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 57s 145ms/step - accuracy: 0.1867 - loss: 3.6254 - val_accuracy: 0.1789 - val_loss: 3.6752 - learning_rate: 0.0010
Epoch 3/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 58s 149ms/step - accuracy: 0.2550 - loss: 3.2227 - val_accuracy: 0.2511 - val_loss: 3.2614 - learning_rate: 0.0010
Epoch 4/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 70s 179ms/step - accuracy: 0.3063 - loss: 2.9525 - val_accuracy: 0.2304 - val_loss: 3.4684 - learning_rate: 0.0010
Epoch 5/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 100s 255ms/step - accuracy: 0.3463 - loss: 2.7316 - val_accuracy: 0.2856 - val_loss: 3.1283 - learning_rate: 0.0010
Epoch 6/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 57s 145ms/step - accuracy: 0.3845 - loss: 2.5573 - val_accuracy: 0.2878 - val_loss: 3.1165 - learning_rate: 0.0010
Epoch 7/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 92s 234ms/step - accuracy: 0.

In [47]:
y_true = val_dataset.labels
y_pred = model.predict(val_dataset).argmax(axis=-1)

40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step


In [ ]:
_, ax = plt.subplots(figsize=(75, 75))
sklearn.metrics.ConfusionMatrixDisplay.from_predictions(y_true, y_pred, ax=ax, colorbar=False)

plt.tight_layout()

In [49]:
np.sum(y_true == y_pred) / len(y_true)

np.float64(0.303)